# Lab 7 · Chuỗi và biểu thức chính quy trên 690 nghìn đánh giá

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 7**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo Bài 7 xử lý cột `name` và `amenities` của bảng dữ liệu chỗ ở (`listings`). Lab này làm việc
với cột **comments** của bảng đánh giá đầy đủ (`reviews`), gồm 690.112 đoạn văn, để xây dựng
các tín hiệu văn bản sẽ được dùng lại ở Bài 11.

**Mục tiêu bài lab**

1. Khảo sát một cột văn bản lớn: độ dài, giá trị thiếu và nội dung ít thông tin.
2. Làm sạch HTML thừa bằng `str.replace` và kiểm chứng bằng đếm.
3. Viết quy tắc nhận diện ngôn ngữ và tín hiệu từ khoá bằng `str.contains`.
4. Dùng `str.extract` trên văn bản tự do và kiểm tra các trường hợp khớp nhầm.

Phiên bản đề: `text-regex-v1` (25/09/2026). Lưu đúng tên **`lab-07.ipynb`**.

## Cách làm và nộp bài

- Phần khởi động (Bài 1–2) và các bài có hướng dẫn: nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết đánh giá các kỹ năng này. Bài tự làm cuối lab được gắn nhãn ✅ mở:
  được dùng AI, kèm trách nhiệm khai báo theo chính sách AI của môn.
- Ghi dự đoán trước khi chạy; điền thân hàm TODO và phần trả lời Markdown.
- Giữ tên hàm, tham số (kể cả giá trị mặc định), cell ID và metadata; viết helper/import thêm trong chính cell bài làm.
  Grader chỉ chạy cell câu đó với input riêng, không phụ thuộc biến `rv`, `c`, `PAT_ES`… ở cell khác.
  `pd` và `np` được cung cấp sẵn khi chấm.
- Public checks chỉ là ví dụ tự kiểm tra trên 16 đoạn đánh giá giả lập; test ẩn dùng văn bản khác cùng hợp đồng
  (có NaN, chuỗi rỗng, chữ hoa, ký tự đặc biệt của regex…).
  Một check lỗi không dừng các check sau. TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa.
- Không sửa Series đầu vào; không ghi cứng kết quả. Không cần xử lý input ngoài hợp đồng.
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1–8, mỗi bài 10 điểm | 80 | Tự động; giảng viên kiểm tra phương pháp |
| Dự đoán và diễn giải (đoạn ngắn, quy tắc ngôn ngữ, khớp nhầm) | 15 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Yêu cầu dùng phương thức `.str` (không vòng lặp Python trên từng dòng), `na=False`, `regex=False` đúng chỗ
được giảng viên kiểm tra cùng code. Bài tự làm ✅ mở không cộng vào 80 điểm tự động.

In [1]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

## Dữ liệu dùng trong lab

Mặc định `USE_SNAPSHOT = False`: notebook dùng **16 đoạn đánh giá giả lập** (`DEMO_COMMENTS`) có đủ các tình huống:
thẻ `<br/>`, giá trị thiếu, chuỗi rỗng, đoạn chỉ có dấu câu, tiếng Tây Ban Nha lẫn tiếng Anh.

Đặt `USE_SNAPSHOT = True` để đọc cột `comments` của file **`data/reviews.csv.gz`** (bảng đầy đủ) của snapshot Santiago
**2026-06-29** — khoảng 690 nghìn dòng, đọc mất cỡ một phút trên Colab. Public checks luôn dùng dữ liệu giả lập.

In [2]:
import numpy as np
import pandas as pd

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/data/reviews.csv.gz")

# 16 đoạn đánh giá GIẢ LẬP, cùng kiểu dữ liệu cột comments của Inside Airbnb.
DEMO_COMMENTS = pd.Series([
    "Excelente ubicación, muy cerca del metro.",
    "Great place near the metro, very clean.",
    "Todo perfecto.<br/>Volvería sin duda.",
    ".",
    None,
    "El departamento está cerca de todo, muy cómodo.",
    "Nice host.<br/>Near Movistar Arena!<br/>Recommended.",
    "Ok",
    "Muy buena estadía cerca de la estación Baquedano.",
    "Lovely apartment, close to everything.",
    "Departamento limpio y cerca de muchos restaurantes.",
    "The departamento was great, near the mall.",
    "Atención excelente, cerca del centro.",
    "WiFi rápido, near the metro station.",
    "",
    "Buena comunicación con el anfitrión.<br/>Recomendado cerca de Movistar Arena.",
], name="comments")

PAT_ES = r"ción|ñ|muy|excelente|departamento"
PAT_GAN = r"(?:cerca de la |cerca del |cerca de |near the |near )([A-Za-zÁ-Úá-úñÑ]+)"

# Đặt USE_SNAPSHOT = True để làm trên 690.112 đánh giá Santiago.
USE_SNAPSHOT = False

if USE_SNAPSHOT:
    rv = pd.read_csv(URL, usecols=["listing_id", "date", "comments"], parse_dates=["date"])
    comments = rv["comments"]
    DATA_LABEL = "Snapshot Santiago 2026-06-29 — cột comments"
else:
    comments = DEMO_COMMENTS.copy()
    DATA_LABEL = "GIẢ LẬP — 16 đoạn đánh giá"

print(DATA_LABEL, "·", len(comments), "dòng")

GIẢ LẬP — 16 đoạn đánh giá · 16 dòng


## Bài 1 · Khởi động: chuẩn hoá trước khi đếm

```python
s = pd.Series(["  Wifi ", "wifi", "WIFI!", "wi-fi"])
```

Dự đoán sau `strip` + `lower` có bao nhiêu phần tử bằng đúng `"wifi"`? Hai biến thể còn lại cần xử lý gì thêm?

### Hợp đồng hàm · 10 điểm tự động

`normalize(s: pd.Series) -> pd.Series`

Nhận Series chuỗi (có thể có NaN). Trả Series mới: bỏ khoảng trắng hai đầu (`str.strip`) rồi chữ thường (`str.lower`);
giá trị thiếu vẫn là NaN; giữ index. Không bỏ dấu câu hay ký tự khác.

In [3]:
def normalize(s: pd.Series) -> pd.Series:
    # TODO: triển khai đúng hợp đồng ở trên.
    return s.str.strip().str.lower()

In [4]:
def check_q1():
    s = pd.Series(["  Wifi ", "wifi", "WIFI!", "wi-fi"])
    result = normalize(s)
    assert isinstance(result, pd.Series)
    assert list(result) == ["wifi", "wifi", "wifi!", "wi-fi"]
    assert (result == "wifi").sum() == 2
    assert list(s) == ["  Wifi ", "wifi", "WIFI!", "wi-fi"], "Không sửa Series đầu vào"
    assert normalize(pd.Series([" A ", None])).isna().tolist() == [False, True]

public_check("Q1 · ví dụ công khai", check_q1)

[ĐẠT] Q1 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


## Bài 2 · Khởi động: bẫy NaN của `contains`

```python
t = pd.Series(["depto centro", None, "casa con vista"])
```

`t.str.contains("centro")` cho gì ở dòng `None`? Vì sao `.sum()` trên kết quả đó dễ gây lỗi hoặc sai?

### Hợp đồng hàm · 10 điểm tự động

`count_contains(s: pd.Series, tu: str) -> int`

Nhận Series chuỗi (có thể có NaN) và một chuỗi con `tu` — có thể chứa ký tự đặc biệt của regex như `.`, `(`, `?`.
Trả **int**: số dòng chứa đúng chuỗi `tu` (so khớp **nguyên văn**, phân biệt hoa thường), giá trị thiếu tính là
**không chứa**. Dùng `str.contains(tu, regex=False, na=False)`.

In [5]:
def count_contains(s: pd.Series, tu: str) -> int:
    # TODO: triển khai đúng hợp đồng ở trên.
    return s.str.contains(tu, regex = False, na = False).sum()

In [6]:
def check_q2():
    t = pd.Series(["depto centro", None, "casa con vista"])
    result = count_contains(t, "centro")
    assert isinstance(result, (int, np.integer)) and result == 1
    assert count_contains(t, "Centro") == 0, "Phân biệt hoa thường"
    assert count_contains(pd.Series(["giá 50.000", "giá 50,000"]), "50.000") == 1, "'.' phải khớp nguyên văn"

public_check("Q2 · ví dụ công khai", check_q2)

[ĐẠT] Q2 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


**Dự đoán và giải thích:** Với dòng `None`, `str.contains("centro")` cho giá trị thiếu; `.sum()` thường bỏ qua giá trị này nhưng không nói rõ quy tắc xử lý. `na=False` quy ước dòng thiếu không chứa từ cần tìm, giúp có mask boolean dùng để đếm và lọc nhất quán. Với chuỗi `"50.000"`, cần `regex=False` vì dấu chấm trong regex khớp một ký tự bất kỳ; so khớp nguyên văn tránh nhận nhầm `"50,000"`.


## Bài 3 · Nhìn tổng quan cột chữ

Bắt đầu bằng ba bước: đếm giá trị thiếu → đo độ dài → kiểm tra một số mẫu ngắn và dài.

### Hợp đồng hàm · 10 điểm tự động

`text_overview(comments: pd.Series, nguong_ngan: int = 20) -> dict`

Nhận Series chuỗi (có thể có NaN, có ít nhất một chuỗi). Trả dict có đúng các khóa:

- `so_nan`: int, số giá trị thiếu; 
- `so_con`: int, số chuỗi còn lại sau khi bỏ NaN;
- `do_dai_tv`: float, trung vị độ dài (`str.len()`) của các chuỗi không thiếu;
- `so_ngan`: int, số chuỗi **không thiếu** có độ dài **nhỏ hơn** `nguong_ngan` (chuỗi rỗng có độ dài 0).

Không sửa input. Đối chiếu snapshot: **36** giá trị thiếu, 690.076 chuỗi, trung vị **115** ký tự,
**58.987** đoạn ngắn hơn 20 ký tự (~8,5%).

In [7]:
def text_overview(comments: pd.Series, nguong_ngan: int = 20) -> dict:
    # 1. Đếm số lượng giá trị thiếu (NaN)
    so_nan = int(comments.isna().sum())
    
    # 2. Lấy danh sách các bình luận hợp lệ (không bị NaN)
    valid_comments = comments.dropna().astype(str)
    
    # 3. Đếm số lượng bình luận còn lại
    so_con = len(valid_comments)
    
    # 4. Tính giá trị trung vị độ dài các bình luận hợp lệ
    do_dai_tv = float(valid_comments.str.len().median()) if so_con > 0 else 0.0
    
    # 5. Đếm số bình luận có độ dài ngắn hơn nguong_ngan (nhỏ hơn, không tính bằng)
    so_ngan = int((valid_comments.str.len() < nguong_ngan).sum())
    
    return {
        'so_nan': so_nan,
        'so_con': so_con,
        'do_dai_tv': do_dai_tv,
        'so_ngan': so_ngan
    }

In [8]:
def check_q3():
    result = text_overview(DEMO_COMMENTS)
    assert set(result) == {"so_nan", "so_con", "do_dai_tv", "so_ngan"}
    assert result["so_nan"] == 1 and result["so_con"] == 15
    assert result["do_dai_tv"] == 39.0
    assert result["so_ngan"] == 3, "'.', 'Ok' và chuỗi rỗng"
    assert text_overview(DEMO_COMMENTS, 40)["so_ngan"] == 8

public_check("Q3 · ví dụ công khai", check_q3)

[ĐẠT] Q3 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [9]:
def thuc_hanh_q3():
    kq = text_overview(comments)
    c = comments.dropna()
    ngan_nhat = c[c.str.len() < 3].head(5).tolist()
    return f"{kq}\nVài đoạn ngắn nhất: {ngan_nhat}"

preview("Bài 3 trên dữ liệu đang dùng", thuc_hanh_q3)

Bài 3 trên dữ liệu đang dùng
{'so_nan': 1, 'so_con': 15, 'do_dai_tv': 39.0, 'so_ngan': 3}
Vài đoạn ngắn nhất: ['.', 'Ok', '']


**Nhận xét:** Các đoạn ngắn nhất trông thế nào?

Từ kết quả chạy thử (['.', 'Ok', '']) và đặc trưng của dữ liệu bình luận thực tế, các đoạn quá ngắn thường chỉ là chuỗi rỗng (""), dấu câu đơn lẻ (".", "!"), biểu tượng cảm xúc, hoặc các từ cảm thán chung chung ("Ok", "Good", "Tốt"). Chúng hầu như không chứa thông tin cụ thể về trải nghiệm hay vấn đề cần phân tích.

Vì sao cần lọc chúng trước khi gửi tới LLM ở Bài 11?

Tiết kiệm chi phí và token: Những bình luận rỗng hoặc vô nghĩa vẫn tiêu tốn token đầu vào và chi phí gọi API nhưng không đem lại giá trị phân tích nào.

Giảm nhiễu và hạn chế LLM suy diễn sai: Khi gặp các đầu vào mơ hồ như "." hoặc chuỗi rỗng, mô hình ngôn ngữ dễ bị nhiễu ngữ cảnh, gán nhãn cảm xúc/chủ đề tùy tiện hoặc tự suy diễn thông tin không có thật.

Tăng tốc độ xử lý: Loại bỏ nhóm bình luận quá ngắn (chiếm khoảng 8,5% trên tập dữ liệu chuẩn) giúp giảm đáng kể lượng dữ liệu phải đưa qua mô hình suy luận.

## Bài 4 · Thẻ HTML `<br/>`

Airbnb lưu xuống dòng thành thẻ `<br/>` ngay trong văn bản. Đếm, thay, rồi **kiểm chứng bằng đếm lại**.

### Hợp đồng hàm · 10 điểm tự động

`clean_br(comments: pd.Series) -> dict`

Nhận Series chuỗi (có thể có NaN). Trả dict có đúng các khóa:

- `so_br`: int, số dòng chứa chuỗi `"<br/>"` (nguyên văn, NaN tính là không chứa);
- `c_sach`: Series mới trong đó **mọi** `"<br/>"` được thay bằng một khoảng trắng `" "` (`regex=False`),
  giá trị thiếu vẫn là NaN, giữ index. Không thay các biến thể khác như `"<br>"`.

Không sửa input. Đối chiếu snapshot: **116.471** đoạn có `<br/>`; sau khi thay còn 0.

In [10]:
def clean_br(comments: pd.Series) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    so_br = int(comments.str.contains("<br/>", case = False, na = False).sum())
    c_sach = comments.str.replace("<br/>"," ", regex = False)
    return {
        'so_br' : so_br,
        'c_sach' : c_sach
    }

In [11]:
def check_q4():
    result = clean_br(DEMO_COMMENTS)
    assert set(result) == {"so_br", "c_sach"}
    assert result["so_br"] == 3
    c_sach = result["c_sach"]
    assert isinstance(c_sach, pd.Series) and len(c_sach) == len(DEMO_COMMENTS)
    assert int(c_sach.str.contains("<br/>", regex=False, na=False).sum()) == 0
    assert c_sach[6] == "Nice host. Near Movistar Arena! Recommended."
    assert c_sach.isna().sum() == 1

public_check("Q4 · ví dụ công khai", check_q4)

[ĐẠT] Q4 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


## Bài 5 · Quy tắc nhận diện ngôn ngữ

Bài giảng dùng một quy tắc xấp xỉ cho tiếng Tây Ban Nha trên cột `name`. Ở đây, ta điều chỉnh bộ từ khoá và áp dụng
cho `comments`: `PAT_ES = r"ción|ñ|muy|excelente|departamento"`.

### Hợp đồng hàm · 10 điểm tự động

`language_flag(comments: pd.Series, pattern: str = PAT_ES) -> dict`

Nhận Series chuỗi (có thể có NaN) và một mẫu regex. Trả dict có đúng các khóa:

- `la_es`: Series **bool** cùng index, `True` nếu chuỗi khớp `pattern` **không phân biệt hoa thường**
  (`str.contains(pattern, case=False, na=False)`); NaN → `False`;
- `ty_le_es`: float, tỷ lệ `True` trên **mọi** dòng của input.

Giá trị mặc định của `pattern` được viết sẵn trong cell bài làm (grader không đọc biến `PAT_ES` ở cell khác).
Đối chiếu snapshot: tỷ lệ ≈ **0,654**.

In [12]:
def language_flag(comments: pd.Series, pattern: str = r"ción|ñ|muy|excelente|departamento") -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    la_es = comments.str.contains(pattern, case = False, na = False)
    ty_le_es = float(la_es.mean())
    return {
        'la_es' : la_es,
        'ty_le_es' : ty_le_es
    }

In [13]:
def check_q5():
    c_sach = DEMO_COMMENTS.dropna().str.replace("<br/>", " ", regex=False)
    result = language_flag(c_sach)
    assert set(result) == {"la_es", "ty_le_es"}
    la_es = result["la_es"]
    assert isinstance(la_es, pd.Series) and la_es.dtype == bool
    assert la_es.sum() == 7 and round(result["ty_le_es"], 3) == 0.467
    assert la_es[11], "'The departamento was great' vẫn bị gán es: quy tắc chỉ là xấp xỉ"
    assert not la_es[2], "'Todo perfecto' không có từ khoá nào"
    assert language_flag(pd.Series(["EXCELENTE", None]))["la_es"].tolist() == [True, False]

public_check("Q5 · ví dụ công khai", check_q5)

[ĐẠT] Q5 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


## Bài 6 · So độ dài hai nhóm

### Hợp đồng hàm · 10 điểm tự động

`median_length_by_flag(comments: pd.Series, flag: pd.Series) -> dict`

Nhận Series chuỗi **không** có NaN và Series bool `flag` **cùng index** (mỗi nhóm có ít nhất một dòng).
Trả dict có đúng các khóa `len_es` (float, trung vị `str.len()` của các dòng `flag == True`) và
`len_khac` (float, trung vị của các dòng còn lại). Chọn dòng bằng mask, không dùng vòng lặp.
Đối chiếu snapshot (sau khi thay `<br/>`): nhóm es **118**, nhóm khác **105** ký tự.

In [14]:
def median_length_by_flag(comments: pd.Series, flag: pd.Series) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    len_es = float(comments.str.len()[flag].median())
    len_khac = float(comments.str.len()[~flag].median())
    return {
        'len_es' : len_es,
        'len_khac' : len_khac
    }

In [15]:
def check_q6():
    c_sach = DEMO_COMMENTS.dropna().str.replace("<br/>", " ", regex=False)
    flag = c_sach.str.contains(PAT_ES, case=False)
    result = median_length_by_flag(c_sach, flag)
    assert set(result) == {"len_es", "len_khac"}
    assert result["len_es"] == 47.0 and result["len_khac"] == 34.5

public_check("Q6 · ví dụ công khai", check_q6)

[ĐẠT] Q6 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [16]:
def thuc_hanh_q4_q6():
    global c_sach
    kq_br = clean_br(comments.dropna())
    c_sach = kq_br["c_sach"]
    kq_es = language_flag(c_sach)
    kq_len = median_length_by_flag(c_sach, kq_es["la_es"])
    return (f"{kq_br['so_br']:,} đoạn có <br/> · tỷ lệ es {kq_es['ty_le_es']:.1%} · "
            f"trung vị độ dài es {kq_len['len_es']:.1f} / khác {kq_len['len_khac']:.1f} ký tự")

preview("Bài 4–6 trên dữ liệu đang dùng", thuc_hanh_q4_q6)

Bài 4–6 trên dữ liệu đang dùng
3 đoạn có <br/> · tỷ lệ es 46.7% · trung vị độ dài es 47.0 / khác 34.5 ký tự


**Giới hạn của quy tắc trên DEMO:** `"The departamento was great, near the mall."` bị gán `es=True` vì chứa từ `departamento`, dù câu chủ yếu là tiếng Anh. `"Todo perfecto.<br/>Volvería sin duda."` không chứa các mẫu `ción|ñ|muy|excelente|departamento`, nên bị bỏ sót dù là tiếng Tây Ban Nha. Quy tắc từ khóa không xét toàn câu; cần kiểm tra mẫu nhầm và bổ sung từ khóa hoặc dùng nhãn tay trước khi suy rộng tỷ lệ ngôn ngữ.


## Bài 7 · `str.extract` trên văn bản tự do

Đoạn đánh giá thường có cấu trúc "cerca de X" / "near X" (gần X). Ta trích **X** để xem các địa điểm thường được nhắc đến:
`PAT_GAN = r"(?:cerca de la |cerca del |cerca de |near the |near )([A-Za-zÁ-Úá-úñÑ]+)"`.

### Hợp đồng hàm · 10 điểm tự động

`extract_near(comments: pd.Series, pattern: str = PAT_GAN) -> dict`

Nhận Series chuỗi (có thể có NaN) và một mẫu regex có **đúng một nhóm bắt**. Trả dict có đúng các khóa:
`gan` — Series kết quả `str.extract(pattern, expand=False)` (lần khớp đầu tiên mỗi dòng, NaN nếu không khớp,
giữ index, giữ nguyên chữ hoa/thường) và `so_trich` — int, số dòng trích được.
Mẫu phân biệt hoa thường như được viết. Đối chiếu snapshot: **33.841** dòng trích được.

In [17]:
def extract_near(comments: pd.Series, pattern: str = r"(?:cerca de la |cerca del |cerca de |near the |near )([A-Za-zÁ-Úá-úñÑ]+)") -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    gan = comments.str.extract(pattern, expand = False) 
    so_trich = int(gan.notna().sum())
    return {
        'gan' : gan,
        'so_trich' : so_trich
    }

In [18]:
def check_q7():
    c_sach = DEMO_COMMENTS.str.replace("<br/>", " ", regex=False)
    result = extract_near(c_sach)
    assert set(result) == {"gan", "so_trich"}
    gan = result["gan"]
    assert isinstance(gan, pd.Series) and len(gan) == len(c_sach)
    assert result["so_trich"] == 9
    assert gan[0] == "metro" and gan[5] == "todo" and gan[15] == "Movistar"
    assert pd.isna(gan[6]), "'Near' viết hoa không khớp 'near ' trong mẫu"

public_check("Q7 · ví dụ công khai", check_q7)

[ĐẠT] Q7 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


## Bài 8 · Bảng tần suất địa điểm và các trường hợp khớp nhầm

"cerca de todo" nghĩa là "gần mọi thứ", không phải địa điểm. Biểu thức chính quy khớp cấu trúc nhưng không hiểu nghĩa;
xem `value_counts()` của kết quả trích xuất giúp phát hiện vấn đề này, rồi lọc bằng danh sách từ dừng.

### Hợp đồng hàm · 10 điểm tự động

`top_places(gan: pd.Series, n: int = 8, bo_qua: tuple = ()) -> pd.Series`

Nhận Series kết quả trích xuất (chuỗi hoặc NaN), số dòng `n >= 1` và các từ cần bỏ qua (viết thường).
Chuyển chữ thường, bỏ NaN, bỏ các từ có trong `bo_qua`, rồi `value_counts().head(n)`:
trả Series index là từ (chữ thường), giá trị là số lần (int), sắp giảm dần. Không sửa input.
Đối chiếu snapshot: `metro` đứng đầu (**7.640** lần); `todo`, `muchos` là khớp nhầm; `movistar` hợp lệ (Movistar Arena).

In [19]:
def top_places(gan: pd.Series, n: int = 8, bo_qua: tuple = ()) -> pd.Series:
    # TODO: triển khai đúng hợp đồng ở trên.
    if n < 1: return 
    s = gan.dropna().str.lower()
    s = s[~s.isin(bo_qua)]
    res = s.value_counts().head(n)
    return res
    

In [20]:
def check_q8():
    gan = pd.Series(["metro", "Metro", None, "todo", "estación", "metro", "Movistar", "muchos", "movistar"])
    result = top_places(gan)
    assert isinstance(result, pd.Series)
    assert result.index[0] == "metro" and result.iloc[0] == 3
    assert result["movistar"] == 2 and len(result) == 5
    loc = top_places(gan, n=2, bo_qua=("todo", "muchos"))
    assert dict(loc) == {"metro": 3, "movistar": 2}
    assert "todo" not in top_places(gan, bo_qua=("todo", "muchos")).index

public_check("Q8 · ví dụ công khai", check_q8)

[ĐẠT] Q8 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [21]:
def thuc_hanh_q7_q8():
    kq = extract_near(c_sach)
    return (f"Trích được {kq['so_trich']:,} dòng\n"
            f"{top_places(kq['gan'])}")

preview("Bài 7–8 trên dữ liệu đang dùng", thuc_hanh_q7_q8)

Bài 7–8 trên dữ liệu đang dùng
Trích được 9 dòng
comments
metro       3
todo        1
estación    1
muchos      1
mall        1
centro      1
movistar    1
Name: count, dtype: int64


**Khớp nhầm:** 

Các từ không phải địa điểm trong bảng tần suất: todo (trong cụm "cerca de todo" – nghĩa là "gần mọi thứ") và muchos (trong cụm "cerca de muchos..." – nghĩa là "gần nhiều..."). (Lưu ý: movistar tuy là tên thương hiệu nhưng được tính là hợp lệ vì viết tắt cho địa điểm Movistar Arena; các từ còn lại như metro, estación, mall, centro đều là danh từ chỉ địa điểm).

Vì sao regex không tự loại được chúng? Biểu thức chính quy (regex) chỉ đối chiếu cấu trúc cú pháp bề mặt (trích xuất từ đứng ngay sau cụm "cerca de"), hoàn toàn không có khả năng hiểu ngữ nghĩa hay nhận biết từ loại (không phân biệt được danh từ chỉ địa điểm với đại từ bất định todo hay lượng từ muchos). Vì các từ này vẫn thỏa mãn đúng quy tắc hình thức đứng sau giới từ nên regex vẫn trích xuất bình thường, buộc phải lọc thủ công bằng danh sách từ dừng (bo_qua) ở bước hậu xử lý.

## Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà) — không cộng vào 80 điểm tự động

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại prompt và cách kiểm chứng.

### Tự làm 1 · Từ khoá hai ngôn ngữ

Xây 2 tín hiệu: `wifi` (mẫu `wifi|internet`) và `parking` (mẫu `parking|estacionamiento`).
Tính **tỷ lệ nhắc đến** của từng tín hiệu trong nhóm es và nhóm còn lại. Nhóm nào quan tâm
parking hơn? Nêu một cách giải thích có thể có và một giới hạn của cách giải thích đó.

### Tự làm 2 · Cải thiện mẫu "cerca de"

Sửa `PAT_GAN` để loại các trường hợp khớp nhầm như "todo"/"muchos" (gợi ý: thêm nhóm loại trừ
hoặc lọc kết quả sau khi trích bằng danh sách từ dừng). Đếm lại 8 giá trị phổ biến nhất — bảng mới sạch hơn
bảng cũ ở chỗ nào? Ghi lại quy trình: sửa mẫu → chạy lại → so kết quả (đúng vòng lặp nới dần của bài giảng).

In [22]:
# 1. Lấy mask nhóm tiếng Tây Ban Nha (la_es) từ kết quả của hàm language_flag
# (Thay DEMO_COMMENTS hoặc comments bằng tên Series bình luận trong notebook của bạn)
flag = language_flag(comments)['la_es']

# 2. Tạo 2 tín hiệu
wifi_flag = comments.str.contains(r"wifi|internet", case=False, na=False)
parking_flag = comments.str.contains(r"parking|estacionamiento", case=False, na=False)

# 3. Tính tỷ lệ nhắc đến trong nhóm 'es' (flag == True)
wifi_rate_es = float(wifi_flag[flag].mean())
parking_rate_es = float(parking_flag[flag].mean())

# 4. Tính tỷ lệ nhắc đến trong nhóm 'còn lại' (flag == False)
wifi_rate_khac = float(wifi_flag[~flag].mean())
parking_rate_khac = float(parking_flag[~flag].mean())

print(f"Nhóm Spanish (es): Wifi = {wifi_rate_es:.4f}, Parking = {parking_rate_es:.4f}")
print(f"Nhóm Còn lại:     Wifi = {wifi_rate_khac:.4f}, Parking = {parking_rate_khac:.4f}")
# Bước 1: Khai báo danh sách từ dừng (stop words) chứa các trường hợp khớp nhầm
BO_QUA_MOI = ("todo", "muchos", "nada", "aqui", "alli", "cualquier")

# Bước 2: Chạy lại trích xuất và lọc bằng top_places
gan = extract_near(comments)["gan"]
top_places_sach = top_places(gan, n=8, bo_qua=BO_QUA_MOI)

print("Bảng 8 địa điểm phổ biến nhất sau khi làm sạch:")
print(top_places_sach)

Nhóm Spanish (es): Wifi = 0.0000, Parking = 0.0000
Nhóm Còn lại:     Wifi = 0.1111, Parking = 0.0000
Bảng 8 địa điểm phổ biến nhất sau khi làm sạch:
comments
metro       3
estación    1
mall        1
centro      1
movistar    1
Name: count, dtype: int64


### Bài tự làm của tôi

#### 1. Từ khoá hai ngôn ngữ (`wifi` và `parking`)

* **Tỷ lệ nhắc đến trên dữ liệu đang dùng:**
  * **Nhóm tiếng Tây Ban Nha (`es`):** `wifi` = **0.0%** (`0.0000`), `parking` = **0.0%** (`0.0000`) *(thay bằng số liệu thực tế nếu chạy trên tập đầy đủ)*
  * **Nhóm còn lại (`khác`):** `wifi` = **11.1%** (`0.1111`), `parking` = **0.0%** (`0.0000`)
* **Nhóm nhắc đến `parking` nhiều hơn:** hai nhóm bằng nhau ở 0/7 và 0/9 trên DEMO; không thể suy ra nhóm nào quan tâm hơn từ mẫu này.
* **Một cách giải thích có thể kiểm tra thêm:** nếu một tập lớn hơn cho tỷ lệ nhắc đến parking khác nhau, có thể so thêm quận, loại phòng và khả năng có bãi đỗ xe; DEMO hiện tại chưa hỗ trợ một giả thuyết về hành vi đi lại của hai nhóm.
* **Một giới hạn của cách giải thích này:**
  1. **Nhiễu từ khoá & ngữ cảnh phủ định:** Regex chỉ đếm sự xuất hiện của từ khoá mà không phân biệt ngữ cảnh khẳng định hay phủ định (ví dụ: *"no hay parking"* – không có chỗ đỗ xe – vẫn bị tính là có nhắc đến `parking`).
  2. **Ranh giới ngôn ngữ chưa chuẩn xác:** Hàm `language_flag` chỉ dựa vào một danh sách từ dừng cố định nên có thể phân loại nhầm người dùng song ngữ hoặc bình luận ngắn, làm sai lệch tỷ lệ giữa hai nhóm.

---

#### 2. Cải thiện mẫu `"cerca de"`

* **Cách sửa quy tắc:**
  * Bổ sung danh sách từ dừng `BO_QUA_MOI = ("todo", "muchos", "nada", "aqui", "alli", "cualquier")` vào tham số `bo_qua` của hàm `top_places` (hoặc thêm nhóm loại trừ phủ định `(?!todo|muchos|nada|aqui|alli|cualquier)` trực tiếp vào `PAT_GAN`) để loại các đại từ/lượng từ không mang nghĩa địa điểm.
* **Bảng mới sạch hơn bảng cũ ở chỗ nào?**
  * Ở bảng cũ, các từ chung chung như `todo` (*"gần mọi thứ"*) và `muchos` (*"gần nhiều..."*) chiếm vị trí trong top đầu dù không phải địa điểm cụ thể.
  * Ở bảng mới, toàn bộ top các từ xuất hiện nhiều nhất (`metro`, `estación`, `mall`, `centro`, `movistar`) đều là **danh từ chỉ địa điểm hoặc công trình thực tế**, phản ánh đúng vị trí xung quanh nơi lưu trú.
* **Quy trình lặp đã thực hiện (`sửa mẫu → chạy lại → so kết quả`):**
  1. **Chạy lần đầu & soi lỗi:** Chạy `top_places(gan)` chưa lọc từ dừng, kiểm tra bảng tần suất và phát hiện các từ khớp nhầm (`todo`, `muchos`).
  2. **Sửa quy tắc:** Thêm các từ nhiễu vào `BO_QUA_MOI` (hoặc nhóm loại trừ trong `PAT_GAN`).
  3. **Chạy lại & so kết quả:** Gọi lại `top_places(gan, n=8, bo_qua=BO_QUA_MOI)` và đối chiếu bảng trước – sau để xác nhận từ nhiễu đã biến mất mà không làm mất các địa điểm hợp lệ như `movistar` (Movistar Arena).

---

#### 3. Ghi chép quy trình dùng AI & kiểm chứng

* **Prompt đã dùng:** *"Hướng dẫn viết đoạn code pandas tạo 2 cờ tín hiệu `wifi|internet` và `parking|estacionamiento`, tính tỷ lệ trung bình theo từng nhóm `la_es`, và lọc từ nhiễu (`todo`, `muchos`) khỏi kết quả `top_places`."*
* **Cách kiểm chứng kết quả:**
  * Dùng `.mean()` trên Series kiểu `bool` (`True = 1`, `False = 0`) để tính trực tiếp tỷ lệ phần trăm, sau đó đối chiếu thủ công số dòng khớp trên tập mẫu 15 dòng (`1/9 = 0.1111` cho `wifi` ở nhóm còn lại).
  * So sánh trực tiếp bảng `value_counts()` trước và sau khi truyền `bo_qua=BO_QUA_MOI` để đảm bảo chỉ loại đúng các từ dừng đã khai báo và giữ nguyên số đếm của các địa điểm thật.
  

## Khai báo sử dụng AI

* **Công cụ đã dùng:** Gemini
* **Bài đã dùng AI và yêu cầu chính:**
  * **Bài tự làm 1 & 2:** Nhờ gợi ý cách viết biểu thức `pandas` để tạo cờ tín hiệu hai ngôn ngữ (`wifi|internet`, `parking|estacionamiento`), tính tỷ lệ theo nhóm `la_es`, và bổ sung danh sách từ dừng `BO_QUA_MOI` để làm sạch kết quả trích xuất sau cụm `"cerca de"`.
* **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:**
  * **Gợi ý đã kiểm chứng:** Dùng `.mean()` trên Series boolean để tính tỷ lệ xuất hiện từ khoá theo nhóm ngôn ngữ, và truyền `bo_qua=BO_QUA_MOI` vào hàm `top_places()` để loại từ nhiễu.
  * **Cách kiểm chứng và kết quả:**
    1. **Đối chiếu phép đếm tỷ lệ `wifi` và `parking`:** Kiểm tra trực tiếp trên các dòng thuộc nhóm còn lại (`~flag`, tổng cộng **9 dòng** trong tập dữ liệu đang chạy): đếm thủ công thấy có đúng **1 dòng** chứa từ khoá `wifi`/`internet` và **0 dòng** chứa `parking`/`estacionamiento`. Phép tính thủ công $1 / 9 \approx 0.1111$ khớp chính xác với kết quả `Wifi = 0.1111, Parking = 0.0000` mà code trả về.
    2. **Đối chiếu bảng tần suất trước và sau khi lọc từ dừng:** Trên **9 dòng** trích xuất được từ mẫu `"cerca de"`, bảng `value_counts()` ban đầu gồm 7 từ (`metro: 3`, `todo: 1`, `estación: 1`, `muchos: 1`, `mall: 1`, `centro: 1`, `movistar: 1`). Sau khi lọc qua `BO_QUA_MOI`, bảng rút gọn còn 5 từ, loại bỏ đúng 2 từ không phải địa điểm (`todo`, `muchos`) và giữ nguyên số đếm của 5 địa điểm hợp lệ (bao gồm `movistar: 1` đại diện cho *Movistar Arena*).

---

## Tóm tắt bài lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| Chuẩn hoá `strip`/`lower`, `na=False`, `regex=False` | mọi bước so khớp chuỗi |
| Khảo sát cột chữ: rỗng, độ dài, mẫu ngắn | lọc đoạn đánh giá trước khi gửi LLM (Bài 11) |
| Làm sạch `<br/>` và kiểm chứng bằng đếm | pipeline xử lý văn bản |
| Quy tắc ngôn ngữ trên 690.112 đoạn đánh giá | baseline cho phần LLM |
| `extract` và kiểm tra trường hợp khớp nhầm | kiểm chứng tín hiệu trích xuất |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

**Bài giảng tiếp theo:** **dữ liệu thời gian** — dùng cột `date` để tổng hợp và so sánh theo thời điểm.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** Nguyễn Tiến Long An / 25022763 / DSE2049_3

1. Giữ tên file **`lab-07.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_07.ipynb`).
2. Hoàn thành hàm TODO, các dự đoán, diễn giải và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
4. Tạo tag `submit/lab-07/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-07` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [23]:
show_public_summary()

PUBLIC CHECKS — không phải điểm chính thức
Q1 · ví dụ công khai: ĐẠT
Q2 · ví dụ công khai: ĐẠT
Q3 · ví dụ công khai: ĐẠT
Q4 · ví dụ công khai: ĐẠT
Q5 · ví dụ công khai: ĐẠT
Q6 · ví dụ công khai: ĐẠT
Q7 · ví dụ công khai: ĐẠT
Q8 · ví dụ công khai: ĐẠT
Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.
